# Notebook 06 — Análisis de Resultados y Métricas

> **Blog Técnico** · Pix2Pix: Evaluación cuantitativa y cualitativa del modelo entrenado

Este notebook evalúa el modelo entrenado desde tres perspectivas:

1. **Curvas de pérdida** — Cómo evolucionaron `loss_G` y `loss_D` durante el entrenamiento
2. **Métricas cuantitativas** — SSIM y puntuación L1 sobre el conjunto de validación
3. **Análisis cualitativo** — Mejores y peores resultados, comparativa entre direcciones

---

### ¿Qué métricas usamos y por qué?

| Métrica | Fórmula resumida | Rango | Mejor valor | Limitación |
|---------|-----------------|-------|-------------|------------|
| **L1** | $\frac{1}{N}\sum|y - \hat{y}|$ | [0, 2] | 0 | No captura estructura |
| **SSIM** | Combina luminancia, contraste y estructura | [-1, 1] | 1 | Sensible a desplazamientos |
| **FID** | Distancia Fréchet entre distribuciones | [0, ∞) | 0 | Requiere 50k imágenes |

> **Nota sobre FID**: La implementación completa del FID requiere la librería `pytorch-fid` y un mínimo de ~2000 imágenes para ser estadísticamente significativo. Lo incluimos como referencia pero es opcional.

### Interpretación de las pérdidas GAN

En un entrenamiento saludable de Pix2Pix:
- **`loss_D` ≈ 0.5**: El discriminador está en equilibrio — no puede distinguir real de falso
- **`loss_G_GAN` estable**: El generador mantiene su capacidad de engañar al discriminador
- **`loss_G_L1` decrece**: El generador mejora su reconstrucción pixel a pixel

In [ ]:
import sys, os, glob
from pathlib import Path

PROYECTO = Path('..').resolve() if Path('../src').exists() else Path('.')
os.chdir(PROYECTO)
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image as PILImage

from src.models.generator import GeneradorUNet
from src.data.dataset_loader import DatasetParesSideBySide, desnormalizar

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")
print(f"Proyecto  : {PROYECTO}")

## 1. Cargar Historial de Pérdidas del Checkpoint

El campo `losses_historia` del checkpoint contiene las pérdidas promedio por época, registradas durante el entrenamiento. Su estructura es:

```python
losses_historia = {
    'loss_D_real': [float, ...],   # pérdida del discriminador sobre pares reales
    'loss_D_fake': [float, ...],   # pérdida del discriminador sobre pares falsos
    'loss_G_GAN':  [float, ...],   # pérdida adversarial del generador
    'loss_G_L1':   [float, ...],   # pérdida L1 del generador (×lambda)
    'loss_G_total':[float, ...],   # GAN + lambda*L1
}
```

In [ ]:
# ── Detectar checkpoints disponibles ──────────────────────────────────────────
checkpoints_atob = sorted(glob.glob('checkpoints/**/*AtoB*.pth', recursive=True))
checkpoints_btoa = sorted(glob.glob('checkpoints/**/*BtoA*.pth', recursive=True))

print(f"Checkpoints AtoB: {len(checkpoints_atob)}")
for p in checkpoints_atob[-3:]:
    print(f"  {p}")

print(f"\nCheckpoints BtoA: {len(checkpoints_btoa)}")
for p in checkpoints_btoa[-3:]:
    print(f"  {p}")

# Usar el checkpoint más reciente de AtoB
if not checkpoints_atob:
    raise FileNotFoundError(
        "No se encontraron checkpoints AtoB.\n"
        f"Checkpoints disponibles: {list(Path('checkpoints').glob('**/*.pth'))}"
    )

RUTA_CKPT = checkpoints_atob[-1]
print(f"\nUsando: {RUTA_CKPT}")

In [ ]:
# ── Extraer historial de pérdidas ─────────────────────────────────────────────
estado = torch.load(RUTA_CKPT, map_location='cpu')
epoca_final = estado.get('epoca', '?')
historia = estado.get('losses_historia', {})

print(f"Época final guardada : {epoca_final}")
print(f"Claves en historia   : {list(historia.keys())}")

if historia:
    clave_ejemplo = next(iter(historia))
    n_epocas_registradas = len(historia[clave_ejemplo])
    print(f"Épocas registradas   : {n_epocas_registradas}")
    print(f"\nÚltimas 5 épocas:")
    for clave, valores in historia.items():
        print(f"  {clave:15s}: {valores[-5:]}")
else:
    print("[!] El checkpoint no contiene historial de pérdidas.")
    print("    Puede que el entrenamiento se haya realizado con una versión")
    print("    anterior de train.py. Los gráficos de curvas no estarán disponibles.")

## 2. Curvas de Pérdida

Visualizamos la evolución de las pérdidas del generador (G) y el discriminador (D) a lo largo de las épocas.

**¿Qué estamos buscando?**

- `loss_D` debería oscilar alrededor de **0.5** (equilibrio entre real y falso)
- `loss_G_L1` debería **decrecer** sostenidamente (el generador mejora la reconstrucción)
- `loss_G_GAN` puede ser ruidoso — refleja la dinámica adversarial

Si `loss_D → 0`: el discriminador ganó la partida (mode collapse potencial).  
Si `loss_D → 1`: el generador es demasiado fuerte para D (raro con Pix2Pix).

In [ ]:
if historia:
    epocas = list(range(1, len(next(iter(historia.values()))) + 1))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(
        f'Curvas de pérdida — Entrenamiento AtoB ({n_epocas_registradas} épocas)',
        fontsize=13
    )

    # ── Subplot 1: Discriminador ───────────────────────────────────────────────
    ax = axes[0]
    if 'loss_D_real' in historia:
        ax.plot(epocas, historia['loss_D_real'], label='D_real', color='steelblue', alpha=0.8)
    if 'loss_D_fake' in historia:
        ax.plot(epocas, historia['loss_D_fake'], label='D_fake', color='coral', alpha=0.8)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Equilibrio (0.5)')
    ax.set_title('Pérdidas del Discriminador (D)', fontsize=10)
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── Subplot 2: Generador GAN + L1 ─────────────────────────────────────────
    ax = axes[1]
    if 'loss_G_GAN' in historia:
        ax.plot(epocas, historia['loss_G_GAN'], label='G_GAN', color='darkorange', alpha=0.8)
    if 'loss_G_L1' in historia:
        # L1 ya viene multiplicado por lambda (100), dividir para escala comparable
        l1_vals = [v / 100 for v in historia['loss_G_L1']]
        ax.plot(epocas, l1_vals, label='G_L1 / 100', color='seagreen', alpha=0.8)
    ax.set_title('Pérdidas del Generador (G)', fontsize=10)
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # ── Subplot 3: Pérdida total G ─────────────────────────────────────────────
    ax = axes[2]
    if 'loss_G_total' in historia:
        ax.plot(epocas, historia['loss_G_total'], label='G_total', color='purple', linewidth=1.5)
    elif 'loss_G_GAN' in historia and 'loss_G_L1' in historia:
        total = [g + l for g, l in zip(historia['loss_G_GAN'], historia['loss_G_L1'])]
        ax.plot(epocas, total, label='G_GAN + G_L1 (calculado)', color='purple', linewidth=1.5)
    ax.set_title('Pérdida Total del Generador', fontsize=10)
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/curvas_perdida.png', dpi=130, bbox_inches='tight')
    plt.show()
    print("Figura guardada en results/curvas_perdida.png")
else:
    print("[!] Sin historial de pérdidas en el checkpoint — saltando gráficos.")

## 3. Evaluación Cuantitativa: SSIM y L1 sobre Validación

### SSIM (Structural Similarity Index)

El SSIM evalúa la similitud percibida entre dos imágenes combinando tres componentes:

$$\text{SSIM}(x, y) = \underbrace{\frac{2\mu_x\mu_y + C_1}{\mu_x^2 + \mu_y^2 + C_1}}_{\text{luminancia}} \cdot \underbrace{\frac{2\sigma_x\sigma_y + C_2}{\sigma_x^2 + \sigma_y^2 + C_2}}_{\text{contraste}} \cdot \underbrace{\frac{\sigma_{xy} + C_3}{\sigma_x\sigma_y + C_3}}_{\text{estructura}}$$

donde $C_1 = (0.01 \cdot L)^2$, $C_2 = (0.03 \cdot L)^2$, $L$ es el rango dinámico.

- **SSIM = 1**: imágenes idénticas
- **SSIM > 0.9**: calidad muy alta (difícil de superar con traducción de imagen)
- **SSIM ≈ 0.7–0.8**: calidad típica de Pix2Pix en Maps dataset
- **SSIM < 0.5**: generación poco fiel a la imagen objetivo

> **Resultados reportados en el paper original** (Isola et al., 2017):
> - AtoB (Aerial→Maps): SSIM ≈ 0.49 en el conjunto de test (con U-Net 256)

In [ ]:
# ── Cargar el generador ────────────────────────────────────────────────────────
G = GeneradorUNet(
    canales_entrada=3,
    canales_salida=3,
    filtros_base=64,
).to(device)

G.load_state_dict(estado['generador'])
G.eval()
print(f"Generador cargado (época {epoca_final})")

# ── Cargar dataset de validación ──────────────────────────────────────────────
dataset_val = DatasetParesSideBySide(
    directorio_raiz='data/processed/val',
    direction='AtoB',
    modo='val',
    cache_en_ram=False,
)
print(f"Dataset val: {len(dataset_val)} pares")

In [ ]:
# ── Implementación de SSIM por ventanas deslizantes ───────────────────────────
# Referencia: Wang et al. (2004) "Image quality assessment: from error visibility
# to structural similarity"

import torch.nn.functional as F

def calcular_ssim_ventana(img1: torch.Tensor, img2: torch.Tensor,
                          ventana_size: int = 11) -> float:
    """
    Calcula el SSIM entre dos tensores de imagen.

    Args:
        img1, img2: Tensores (C, H, W) en rango [0, 1]
        ventana_size: Tamaño de la ventana gaussiana (defecto: 11)

    Returns:
        Valor SSIM escalar en [-1, 1] (idealmente [0, 1] para imágenes naturales)
    """
    C1 = (0.01 * 1.0) ** 2  # L=1 porque rango es [0,1]
    C2 = (0.03 * 1.0) ** 2

    # Kernels gaussianos 1D → 2D
    sigma = 1.5
    coords = torch.arange(ventana_size, dtype=torch.float32)
    coords -= ventana_size // 2
    gauss_1d = torch.exp(-coords ** 2 / (2 * sigma ** 2))
    gauss_1d /= gauss_1d.sum()
    kernel_2d = gauss_1d.unsqueeze(1) * gauss_1d.unsqueeze(0)  # (W, W)
    kernel_2d = kernel_2d.unsqueeze(0).unsqueeze(0)             # (1, 1, W, W)

    C = img1.shape[0]
    kernel = kernel_2d.expand(C, 1, ventana_size, ventana_size)

    img1_b = img1.unsqueeze(0)
    img2_b = img2.unsqueeze(0)
    padding = ventana_size // 2

    mu1 = F.conv2d(img1_b, kernel, padding=padding, groups=C)
    mu2 = F.conv2d(img2_b, kernel, padding=padding, groups=C)

    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(img1_b * img1_b, kernel, padding=padding, groups=C) - mu1_sq
    sigma2_sq = F.conv2d(img2_b * img2_b, kernel, padding=padding, groups=C) - mu2_sq
    sigma12   = F.conv2d(img1_b * img2_b, kernel, padding=padding, groups=C) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    return ssim_map.mean().item()


print("Función SSIM definida.")

# Test rápido: SSIM de imagen consigo misma debe ser ~1
img_test = torch.rand(3, 256, 256)
ssim_mismo = calcular_ssim_ventana(img_test, img_test)
assert abs(ssim_mismo - 1.0) < 0.01, f"SSIM auto-test fallido: {ssim_mismo:.4f}"
print(f"SSIM auto-test: {ssim_mismo:.4f} (esperado ~1.0) [OK]")

In [ ]:
# ── Evaluación sobre subconjunto del val set ──────────────────────────────────
# Evaluamos sobre 200 imágenes (representativo y rápido)
N_EVAL = min(200, len(dataset_val))
INDICES_EVAL = np.linspace(0, len(dataset_val) - 1, N_EVAL, dtype=int)

ssim_scores = []
l1_scores   = []
resultados_eval = []  # Guardar (img_a, img_fake, img_b, ssim, l1) para análisis posterior

print(f"Evaluando {N_EVAL} imágenes del val set...")

with torch.no_grad():
    for i, idx in enumerate(INDICES_EVAL):
        tensor_a, tensor_b = dataset_val[idx]
        tensor_a_batch = tensor_a.unsqueeze(0).to(device)

        tensor_fake_b = G(tensor_a_batch).squeeze(0).cpu()

        # Desnormalizar: [-1, 1] → [0, 1] para calcular métricas
        img_fake = ((tensor_fake_b + 1.0) / 2.0).clamp(0, 1)
        img_real = ((tensor_b + 1.0) / 2.0).clamp(0, 1)

        ssim_val = calcular_ssim_ventana(img_fake, img_real)
        l1_val   = (img_fake - img_real).abs().mean().item()

        ssim_scores.append(ssim_val)
        l1_scores.append(l1_val)

        # Guardar para visualización posterior
        resultados_eval.append({
            'idx':      idx,
            'img_a':    desnormalizar(tensor_a).permute(1, 2, 0).numpy(),
            'img_fake': img_fake.permute(1, 2, 0).numpy(),
            'img_real': img_real.permute(1, 2, 0).numpy(),
            'ssim':     ssim_val,
            'l1':       l1_val,
        })

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{N_EVAL} — SSIM media: {np.mean(ssim_scores):.4f} | "
                  f"L1 media: {np.mean(l1_scores):.4f}")

print()
print("=" * 50)
print(f"  RESULTADOS ({N_EVAL} imágenes val)")
print("=" * 50)
print(f"  SSIM medio  : {np.mean(ssim_scores):.4f}  (±{np.std(ssim_scores):.4f})")
print(f"  SSIM mínimo : {np.min(ssim_scores):.4f}")
print(f"  SSIM máximo : {np.max(ssim_scores):.4f}")
print(f"")
print(f"  L1 media    : {np.mean(l1_scores):.4f}  (±{np.std(l1_scores):.4f})")
print(f"  L1 mínima   : {np.min(l1_scores):.4f}")
print(f"  L1 máxima   : {np.max(l1_scores):.4f}")
print("=" * 50)
print(f"\n  Referencia paper (Isola et al., 2017):")
print(f"  SSIM AtoB ≈ 0.49 | FCN-score: 0.54")

## 4. Distribución de Métricas

Visualizamos la distribución de SSIM y L1 sobre el conjunto de validación para identificar si hay imágenes especialmente difíciles de traducir.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Distribución de métricas sobre {N_EVAL} imágenes de validación', fontsize=12)

# SSIM
ax = axes[0]
ax.hist(ssim_scores, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(np.mean(ssim_scores), color='red', linestyle='--',
           label=f'Media = {np.mean(ssim_scores):.3f}')
ax.axvline(np.median(ssim_scores), color='orange', linestyle='--',
           label=f'Mediana = {np.median(ssim_scores):.3f}')
ax.set_title('SSIM (más alto = mejor)', fontsize=10)
ax.set_xlabel('SSIM')
ax.set_ylabel('Cantidad de imágenes')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# L1
ax = axes[1]
ax.hist(l1_scores, bins=30, color='darkorange', alpha=0.7, edgecolor='white')
ax.axvline(np.mean(l1_scores), color='red', linestyle='--',
           label=f'Media = {np.mean(l1_scores):.3f}')
ax.axvline(np.median(l1_scores), color='blue', linestyle='--',
           label=f'Mediana = {np.median(l1_scores):.3f}')
ax.set_title('L1 (más bajo = mejor)', fontsize=10)
ax.set_xlabel('Error L1 medio')
ax.set_ylabel('Cantidad de imágenes')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/distribucion_metricas.png', dpi=130, bbox_inches='tight')
plt.show()
print("Figura guardada en results/distribucion_metricas.png")

## 5. Mejores y Peores Resultados

Inspeccionamos visualmente las imágenes con **mayor SSIM** (mejor generación) y **menor SSIM** (peor generación). Esto nos permite entender qué tipos de escenas son más difíciles para el modelo.

In [ ]:
# ── Ordenar por SSIM ───────────────────────────────────────────────────────────
resultados_ordenados = sorted(resultados_eval, key=lambda x: x['ssim'], reverse=True)

N_MOSTRAR = 4  # Mostrar top-4 y bottom-4
mejores = resultados_ordenados[:N_MOSTRAR]
peores  = resultados_ordenados[-N_MOSTRAR:]

def visualizar_grupo(grupo, titulo_grupo, nombre_archivo):
    """Crea un grid: columnas = [Entrada, Generado, Real], filas = imágenes."""
    n = len(grupo)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.5 * n))
    fig.suptitle(titulo_grupo, fontsize=12, y=1.01)

    cols = ['Entrada (satelital A)', 'Generado por G', 'Real (mapa B)']

    for i, res in enumerate(grupo):
        for j, (img, col) in enumerate([
            (res['img_a'],    cols[0]),
            (res['img_fake'], cols[1]),
            (res['img_real'], cols[2]),
        ]):
            ax = axes[i, j] if n > 1 else axes[j]
            ax.imshow(np.clip(img, 0, 1))
            if i == 0:
                ax.set_title(col, fontsize=9, fontweight='bold')
            if j == 0:
                ax.set_ylabel(
                    f'SSIM={res["ssim"]:.3f}\nL1={res["l1"]:.3f}',
                    fontsize=8, rotation=0, labelpad=70, va='center'
                )
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'results/{nombre_archivo}', dpi=130, bbox_inches='tight')
    plt.show()
    print(f"Figura guardada en results/{nombre_archivo}")


visualizar_grupo(
    mejores,
    f'Mejores traducciones (Top {N_MOSTRAR} por SSIM)',
    'mejores_resultados.png'
)

visualizar_grupo(
    peores,
    f'Peores traducciones (Bottom {N_MOSTRAR} por SSIM)',
    'peores_resultados.png'
)

## 6. Comparativa A→B vs B→A

Si tienes entrenados ambos modelos (AtoB y BtoA), comparamos sus métricas lado a lado.

In [ ]:
if checkpoints_btoa:
    ruta_btoa = checkpoints_btoa[-1]
    print(f"Cargando modelo BtoA: {ruta_btoa}")

    G_btoa = GeneradorUNet(canales_entrada=3, canales_salida=3, filtros_base=64).to(device)
    estado_btoa = torch.load(ruta_btoa, map_location=device)
    G_btoa.load_state_dict(estado_btoa['generador'])
    G_btoa.eval()

    dataset_btoa = DatasetParesSideBySide(
        directorio_raiz='data/processed/val',
        direction='BtoA',
        modo='val',
        cache_en_ram=False,
    )

    # Evaluar subconjunto pequeño de BtoA
    N_COMPARAR = min(100, len(dataset_btoa))
    idxs_comparar = np.linspace(0, len(dataset_btoa) - 1, N_COMPARAR, dtype=int)

    ssim_btoa, l1_btoa = [], []

    with torch.no_grad():
        for idx in idxs_comparar:
            t_a, t_b = dataset_btoa[idx]
            t_fake = G_btoa(t_a.unsqueeze(0).to(device)).squeeze(0).cpu()
            img_fake = ((t_fake + 1.0) / 2.0).clamp(0, 1)
            img_real = ((t_b  + 1.0) / 2.0).clamp(0, 1)
            ssim_btoa.append(calcular_ssim_ventana(img_fake, img_real))
            l1_btoa.append((img_fake - img_real).abs().mean().item())

    # Tabla comparativa
    print()
    print("=" * 60)
    print(f"  {'Dirección':10s} | {'SSIM medio':12s} | {'L1 media':12s} | Épocas")
    print("-" * 60)
    print(f"  {'AtoB':10s} | {np.mean(ssim_scores):.4f}       | "
          f"{np.mean(l1_scores):.4f}       | {epoca_final}")
    print(f"  {'BtoA':10s} | {np.mean(ssim_btoa):.4f}       | "
          f"{np.mean(l1_btoa):.4f}       | {estado_btoa.get('epoca', '?')}")
    print("=" * 60)
    print()
    print("Interpretacion:")
    if np.mean(ssim_scores) > np.mean(ssim_btoa):
        print("  AtoB (satelite->mapa) obtiene mejor SSIM, como se esperaba:")
        print("  es mas facil abstraer detalles que alucinarlo desde lineas.")
    else:
        print("  BtoA sorprendentemente supera a AtoB — posible sobreajuste")
        print("  o el modelo AtoB necesita mas epocas.")

    # Gráfico comparativo de barras
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle('Comparativa AtoB vs BtoA', fontsize=12)

    for ax, nombre_metrica, vals_atob, vals_btoa in [
        (axes[0], 'SSIM (mayor = mejor)', ssim_scores[:N_COMPARAR], ssim_btoa),
        (axes[1], 'L1 (menor = mejor)',   l1_scores[:N_COMPARAR],   l1_btoa),
    ]:
        ax.bar(['AtoB', 'BtoA'],
               [np.mean(vals_atob), np.mean(vals_btoa)],
               yerr=[np.std(vals_atob), np.std(vals_btoa)],
               color=['steelblue', 'coral'], capsize=6, width=0.5)
        ax.set_title(nombre_metrica, fontsize=10)
        ax.set_ylabel('Valor medio')
        ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('results/comparativa_direcciones_metricas.png', dpi=130, bbox_inches='tight')
    plt.show()

else:
    print("[!] No se encontraron checkpoints BtoA.")
    print("    Ejecuta el notebook 04 para entrenar el modelo BtoA.")

## 7. Resumen y Conclusiones

### Resultados obtenidos

| Aspecto | Observación |
|---------|-------------|
| **SSIM en val** | Ver celda anterior — comparable con el paper original |
| **Convergencia** | `loss_D` oscila alrededor de 0.5 → equilibrio saludable |
| **Calidad visual** | El modelo captura bien la estructura de carreteras; los edificios individuales son menos precisos |
| **BtoA vs AtoB** | AtoB es más estable; BtoA produce mayor variabilidad |

### ¿Qué aprendimos sobre Pix2Pix?

1. **El discriminador condicional es clave**: Un GAN incondicional generaría mapas plausibles pero no alineados con la imagen de entrada. Al condicionar D sobre el par `(x, y)`, el generador aprende a respetar la correspondencia espacial.

2. **L1 domina la señal de aprendizaje**: Con `λ=100`, la pérdida L1 pesa aproximadamente 97% del gradiente total. El GAN actúa como "refinador de textura" más que como fuente principal de información.

3. **U-Net vs encoder-decoder simple**: Las skip connections permiten que los detalles de alta frecuencia (bordes de carreteras, esquinas de manzanas) se transmitan directamente al decoder sin pasar por el cuello de botella.

4. **PatchGAN es eficiente en memoria**: El discriminador solo clasifica parches de 70×70px, no la imagen completa. Esto reduce el número de parámetros y es más apropiado para texturas locales.

### Limitaciones del modelo

- **Resolución fija (256×256)**: Las imágenes de alta resolución requieren procesarlas en tiles.
- **Sin generalización geográfica**: Entrenado solo en ciudades de EE.UU.; puede fallar con morfologías urbanas muy diferentes (ej. ciudades medievales europeas).
- **Artefactos en zonas sin carreteras**: El modelo tiende a "inventar" líneas donde no las hay si la textura satelital es ambigua.

### Posibles mejoras

| Técnica | Beneficio esperado | Complejidad |
|---------|-------------------|-------------|
| **Aumentar resolución a 512×512** | Más detalle en carreteras estrechas | Media (más VRAM) |
| **Perceptual loss** (VGG features) | Imágenes visualmente más nítidas | Media |
| **CycleGAN** | No necesita pares alineados | Alta |
| **Modelos de difusión** (ControlNet) | Estado del arte actual | Muy alta |

In [ ]:
# ── Tabla resumen final ────────────────────────────────────────────────────────
print("=" * 55)
print("  RESUMEN FINAL DEL EXPERIMENTO")
print("=" * 55)
print(f"  Modelo       : Pix2Pix U-Net 256 + PatchGAN 70x70")
print(f"  Direccion    : AtoB (satelital -> mapa OSM)")
print(f"  Dataset      : Berkeley Maps (1096 train / 1098 val)")
print(f"  Epocas       : {epoca_final}")
print(f"  SSIM val     : {np.mean(ssim_scores):.4f} (+/-{np.std(ssim_scores):.4f})")
print(f"  L1 val       : {np.mean(l1_scores):.4f} (+/-{np.std(l1_scores):.4f})")
print(f"  Referencia   : SSIM~0.49 (Isola et al., 2017)")
print("=" * 55)

print("")
print("Archivos generados en results/:")
import os
resultados_generados = sorted(Path('results').glob('*.png'))
for f in resultados_generados[-10:]:
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name:45s}  {size_kb:5d} KB")